**Data Ingestion(Loading)** → Text Splitter → Embeddings → Vector Store → Retriever → LLM Output

-> uv pip install langchain-community - install

**Data Ingestion**
Data Ingestion means loading external information (like PDFs, websites, or text files) into memory so that an LLM (Large Language Model) can understand and answer questions based on that data.

In [ ]:
#data loading
from langchain_community.document_loaders import TextLoader 
text_loader = TextLoader('text.txt')
print(text_loader.load())

In [ ]:
#loading a pdf file
from langchain_community.document_loaders import PyPDFLoader
pdf_loader = PyPDFLoader("Mastering AI Agents-compressed.pdf")
print(pdf_loader.load())

In [ ]:
#loading a axive file
from langchain_community.document_loaders import ArxivLoader
arxiv_loader = ArxivLoader(query="2603.19191")
print(arxiv_loader.load())

In [ ]:
#loading from wikipedia
from langchain_community.document_loaders import WikipediaLoader
wikipedia_loader = WikipediaLoader(query="AI agents")   
print(wikipedia_loader.load())

# Text Splitting

Data Ingestion(Loading) → **Text Splitter** → Embeddings → Vector Store → Retriever → LLM Output

uv pip install -U langchain-text-splitters

LLMs like GPT-4 can only process a limited number of tokens (~4,000–32,000 depending on the model). So, large documents must be split into smaller, meaningful chunks.

# Types
# CharacterTextSplitter 
    - Best for basic splitting based on character limits.
# RecursiveCharacterTextSplitter 
    - Smart splitter that tries different strategies (like paragraphs, sentences, words) recursively for best results.

Term	        Meaning
chunk_size	    Max number of characters/tokens per chunk
chunk_overlap	Number of characters/tokens repeated between adjacent chunks (helps preserve context continuity)
                (1 2) - 1
                    (2 3) - 2
                        (3 4) - 3

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)
texts = text_splitter.split_documents(pdf_loader.load())
for i, text in enumerate(texts):
    print(f"Chunk {i+1}: {text.page_content}\n---\n")

# Embeddings

Data Ingestion(Loading) → Text Splitter → **Embeddings** → Vector Store → Retriever → LLM Output

    - Embeddings = converting text into numbers (vectors)
    - So machines can understand meaning

# Example

    - “I love dogs” → [0.21, -0.45, 0.88, ...]
    - “I like puppies” → [0.20, -0.40, 0.85, ...]
    -  Both vectors are close → meaning is similar

# Key Idea

    - Similar meaning → vectors are close together
    - Different meaning → vectors are far apart

**OpenAI Embeddings** → For scalable cloud systems, production apps (like resume matchers) - (Open API Key)
**HuggingFace Embeddings** → For fast local testing, prototyping, or privacy-sensitive tasks

- pip install langchain openai sentence-transformers

In [ ]:
#open ai embeddings
import os
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(openai_api_key=os.environ.get("OPENAI_API_KEY"))
data = embeddings.embed_query("What is an AI agent?")
print(data[:5])


In [ ]:
#Emabeddings using hugging face
from langchain_huggingface import HuggingFaceEmbeddings
huggingface_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
data = huggingface_embeddings.embed_query("What is an AI agent?")
print(data[:5])


**Vector Store**
Data Ingestion(Loading) → Text Splitter → Embeddings → **Vector Store** → Retriever → LLM Output

uv pip install faiss-cpu

What is a Vector Store?
Core Idea
Vector store = database for storing embeddings (vectors)
Used to search based on meaning (similarity)



In [ ]:
import os
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

#Embeddings
embeddings = OpenAIEmbeddings(openai_api_key=os.environ.get("OPENAI_API_KEY"))

#data
Text = ["What is an AI agent?", 
        "How do AI agents work?", 
        "What are the applications of AI agents?", 
        "What are the challenges in developing AI agents?", 
        "What is the future of AI agents?"]

#vector store
vector_store = FAISS.from_texts(Text, embedding=embeddings)

vector_store.save_local("faiss_index")


**Retriever**
Data Ingestion(Loading) → Text Splitter → Embeddings → Vector Store → **Retriever** → LLM Output

In [3]:
#Retrieval
retriever = vector_store.as_retriever()
query = "What is an AI agent?"
docs = retriever.invoke(query)
for doc in docs:
    print(doc.page_content)

What is an AI agent?
How do AI agents work?
What is the future of AI agents?
What are the applications of AI agents?
